In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from sklearn.svm import OneClassSVM

In [2]:
data = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")
data.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_usd_transaction_value,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity


In [3]:
print("Number of rows:", data.shape[0])
print("Number of columns:", data.shape[1])

print("Missing values:", data.isnull().sum().sum())
print("Duplicated rows:", data.duplicated().sum())

Number of rows: 214
Number of columns: 50
Missing values: 0
Duplicated rows: 0


In [4]:
data["report_month"] = pd.to_datetime(data["report_month"],format="%Y-%m")
data["report_month_number"] = data["report_month"].dt.month
data = data.sort_values("report_month").reset_index(drop=True)
data[["report_month", "report_year", "report_month_number"]].head()

,report_month,report_year,report_month_number
0,2022-11-01,2022,11
1,2022-11-01,2022,11
2,2022-11-01,2022,11
3,2022-12-01,2022,12
4,2022-12-01,2022,12


In [5]:
num_cols = [
    "report_year",
    "report_month_number",
    "users_total",
    "transactions_total",
    "customer_legal_entity_count",
    "wallet_individual_count",
    "user_new_count",
    "tx_purchases_count",
    "tx_money_transfers_count",
    "ef_digital_payment_count",
    "ach_jod_transaction_count",
    "returned_cheque_count"
]

cat_cols = ["system"]

In [6]:
data[num_cols + cat_cols].head()

,report_year,report_month_number,users_total,transactions_total,customer_legal_entity_count,wallet_individual_count,user_new_count,tx_purchases_count,tx_money_transfers_count,ef_digital_payment_count,ach_jod_transaction_count,returned_cheque_count,system
0,2022,11,523000,985000,4300,0,0,84300,900800,0,0,0,CliQ
1,2022,11,2020000,1260000,0,2360000,0,58000,1053000,0,0,0,JoMoPay
2,2022,11,3580000,3630000,0,0,35000,0,0,2850000,0,0,eFAWATEERcom
3,2022,12,561000,1180000,4900,0,0,91900,1091000,0,0,0,CliQ
4,2022,12,2050000,1510000,0,2400000,0,62000,1221000,0,0,0,JoMoPay


In [7]:
ohe = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
X_cat = ohe.fit_transform(data[cat_cols])

In [8]:
ohe.get_feature_names_out(cat_cols)

array(['system_ACH', 'system_CliQ', 'system_ECCU', 'system_JoMoPay',
       'system_eFAWATEERcom'], dtype=object)

In [9]:
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(data[num_cols])

In [10]:
X = np.hstack([X_num_scaled, X_cat])
print("Final X shape:", X.shape)

Final X shape: (214, 17)


In [11]:
feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 17
['report_year', 'report_month_number', 'users_total', 'transactions_total', 'customer_legal_entity_count', 'wallet_individual_count', 'user_new_count', 'tx_purchases_count', 'tx_money_transfers_count', 'ef_digital_payment_count', 'ach_jod_transaction_count', 'returned_cheque_count', 'system_ACH', 'system_CliQ', 'system_ECCU', 'system_JoMoPay', 'system_eFAWATEERcom']


In [12]:
pickle.dump(scaler, open("pkl_files/scaler.pkl", "wb"))
pickle.dump(ohe, open("pkl_files/onehot_encoder.pkl", "wb"))
pickle.dump(num_cols, open("pkl_files/num_cols.pkl", "wb"))
pickle.dump(cat_cols, open("pkl_files/cat_cols.pkl", "wb"))
pickle.dump(feature_names, open("pkl_files/feature_names.pkl", "wb"))

print("Preprocessing objects saved successfully")

Preprocessing objects saved successfully


Isolation Forest

In [13]:
iso = IsolationForest(
    contamination=0.03,
    n_estimators=300,
    max_samples="auto",
    max_features=1.0,
    random_state=42
)

iso.fit(X)

IsolationForest(contamination=0.03, n_estimators=300, random_state=42)

In [14]:
data["score_iso"] = -iso.decision_function(X)
data[["system", "score_iso"]].head()

,system,score_iso
0,CliQ,-0.036295
1,JoMoPay,-0.054880
2,eFAWATEERcom,-0.020277
3,CliQ,-0.040189
4,JoMoPay,-0.052718


In [15]:
threshold = np.percentile(data["score_iso"], 97)
data["anomaly_iso"] = (data["score_iso"] >= threshold).astype(int)
data[["system", "score_iso", "anomaly_iso"]].head()

,system,score_iso,anomaly_iso
0,CliQ,-0.036295,0
1,JoMoPay,-0.054880,0
2,eFAWATEERcom,-0.020277,0
3,CliQ,-0.040189,0
4,JoMoPay,-0.052718,0


In [28]:
ranked_anomalies = (data[data["anomaly_iso"] == 1].sort_values("score_iso", ascending=False)
    [["report_month", "system", "score_iso", "anomaly_iso"]].copy()
)

ranked_anomalies["report_month"] = ranked_anomalies["report_month"].astype(str)
ranked_anomalies["reason"] = ranked_anomalies["score_iso"].apply(lambda score:f"Anomaly score {score:.4f} exceeded the threshold {threshold:.4f}.")
ranked_anomalies.to_pickle("pkl_files/ranked_anomalies.pkl")

In [17]:
data["anomaly_iso"].value_counts()

anomaly_iso
0    207
1      7
Name: count, dtype: int64

In [18]:
data_synth = data.copy()
idx = data_synth.sample(frac=0.03,random_state=42).index

data_synth.loc[idx, "transactions_total"] = (data_synth["transactions_total"] * 5)
data_synth.loc[idx, "users_total"] = (data_synth["users_total"] * 5)

data_synth["is_synthetic"] = 0
data_synth.loc[idx, "is_synthetic"] = 1

data_synth[["transactions_total","users_total","is_synthetic"]].head()

,transactions_total,users_total,is_synthetic
0,985000,523000,0
1,1260000,2020000,0
2,3630000,3580000,0
3,1180000,561000,0
4,1510000,2050000,0


In [19]:
X_num_synth = scaler.transform(data_synth[num_cols])
X_cat_synth = ohe.transform(data_synth[cat_cols])
X_synth = np.hstack([X_num_synth,X_cat_synth])

scores_iso_synth = -iso.decision_function(X_synth)
iso_roc_auc = roc_auc_score(data_synth["is_synthetic"],scores_iso_synth)

print("Isolation Forest ROC-AUC:", iso_roc_auc)

Isolation Forest ROC-AUC: 0.8149038461538463


One-Class SVM

In [20]:
for g in ["scale", 0.01, 0.1]:
    
    oc = OneClassSVM(kernel="rbf", nu=0.03,gamma=g)
    oc.fit(X)
    scores_ocsvm_synth = -oc.decision_function(X_synth)
    roc = roc_auc_score(data_synth["is_synthetic"],scores_ocsvm_synth)
    print("gamma:", g,"ROC-AUC:", roc)

gamma: scale ROC-AUC: 0.703525641025641
gamma: 0.01 ROC-AUC: 0.7267628205128205
gamma: 0.1 ROC-AUC: 0.733974358974359


In [21]:
ocsvm = OneClassSVM(
    kernel="rbf",
    nu=0.03,
    gamma= 0.1
)

ocsvm.fit(X)

OneClassSVM(gamma=0.1, nu=0.03)

In [22]:
data["score_ocsvm"] = -ocsvm.decision_function(X)
data["anomaly_ocsvm"] = (ocsvm.predict(X) == -1).astype(int)
data[["system","score_ocsvm","anomaly_ocsvm"]].head()

,system,score_ocsvm,anomaly_ocsvm
0,CliQ,-0.019955,0
1,JoMoPay,-0.027682,0
2,eFAWATEERcom,-0.021327,0
3,CliQ,0.000154,1
4,JoMoPay,0.000206,1


In [23]:
data["anomaly_ocsvm"].value_counts()

anomaly_ocsvm
0    199
1     15
Name: count, dtype: int64

In [24]:
scores_ocsvm_synth = -ocsvm.decision_function(X_synth)
ocsvm_roc_auc = roc_auc_score(data_synth["is_synthetic"],scores_ocsvm_synth)
print("One-Class SVM ROC-AUC:", ocsvm_roc_auc)

One-Class SVM ROC-AUC: 0.733974358974359


In [25]:
comparison_results = pd.DataFrame({
    "Model": ["Isolation Forest","One-Class SVM"],
    "ROC-AUC": [iso_roc_auc, ocsvm_roc_auc],
    "Detected Anomalies": [data["anomaly_iso"].sum(),data["anomaly_ocsvm"].sum()]
})
comparison_results

,Model,ROC-AUC,Detected Anomalies
0,Isolation Forest,0.814904,7
1,One-Class SVM,0.733974,15


Isolation Forest is the best 

In [26]:
best_anomaly_model = iso

pickle.dump(best_anomaly_model, open("pkl_files/best_anomaly_model.pkl", "wb"))
pickle.dump(threshold, open("pkl_files/anomaly_threshold.pkl", "wb"))

print("Best anomaly model saved successfully")
print("Selected model: Isolation Forest")
print("Anomaly threshold:", threshold)

Best anomaly model saved successfully
Selected model: Isolation Forest
Anomaly threshold: -1.3660947373317356e-17


In [27]:
system_reference_stats = {}

for system_name, system_data in data.groupby("system"):
    system_reference_stats[system_name] = {}
    for feature in num_cols:
        system_reference_stats[system_name][feature] = {
            "mean": float(system_data[feature].mean()),
            "std": float(system_data[feature].std()),
            "min": float(system_data[feature].min()),
            "max": float(system_data[feature].max())
        }

pickle.dump(system_reference_stats, open("pkl_files/system_reference_stats.pkl", "wb"))